## Initial Settings

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import time
import os
import sys
import math
import random
import numpy as np
import cv2
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as widgets
from pynq import Overlay, MMIO, PL, allocate
from pynq.lib.video import *
from pynq_dpu import DpuOverlay
import spidev
import colorsys


# load_model 줄을 이걸로 교체 (주석 풀고)
overlay = DpuOverlay("/home/xilinx/jupyter_notebooks/KGIC/driveCode/dpu/dpu.bit")   # DPU+PWM 비트스트림 (data_collection에서 검증된 경로)
overlay.load_model("/home/xilinx/jupyter_notebooks/KGIC/modelCode/tiny-yolov3_256.xmodel")


# # Load YOLOv3
# overlay.load_model("../xmodel/top-tiny-yolov3_coco_256.xmodel")

print("init complet")

init complet


## 모터 초기설정

In [2]:
# 모터 설정
motor_0_address = 0x00A0000000
motor_1_address = 0x00A0010000
motor_2_address = 0x00A0020000
motor_3_address = 0x00A0030000
motor_4_address = 0x00A0040000
motor_5_address = 0x00A0050000

address_range = 0x10000

motor_0 = MMIO(motor_0_address , address_range)
motor_1 = MMIO(motor_1_address , address_range) # CHANGE 
motor_2 = MMIO(motor_2_address , address_range)
motor_3 = MMIO(motor_3_address , address_range)
motor_4 = MMIO(motor_4_address , address_range) # CHANGE
motor_5 = MMIO(motor_5_address , address_range)

print("init complet")

init complet


In [3]:
# 모터 기본 설정
# size = 2ms ==3.33ns x 600600.600..... 
# about 600600
size = 600600
motor_0_percent = 50/100
motor_1_percent = 50/100
motor_2_percent = 50/100
motor_3_percent = 50/100
motor_4_percent = 50/100
motor_5_percent = 50/100

motor_0_duty = size * motor_0_percent
motor_1_duty = size * motor_1_percent
motor_2_duty = size * motor_2_percent
motor_3_duty = size * motor_3_percent
motor_4_duty = size * motor_4_percent
motor_5_duty = size * motor_5_percent
print(int(motor_0_duty))


motor_0.write(0x00,size)           # size
motor_0.write(0x04,int(motor_0_duty))   # DUTY
motor_0.write(0x08,0)              # valid

motor_1.write(0x00,size)  # size
motor_1.write(0x04,int(motor_1_duty))   # DUTY
motor_1.write(0x08,0)       # valid

motor_2.write(0x00,size)  # size
motor_2.write(0x04,int(motor_2_duty))   # DUTY
motor_2.write(0x08,0)       # valid

motor_3.write(0x00,size)  # size
motor_3.write(0x04,int(motor_3_duty))   # DUTY
motor_3.write(0x08,0)       # valid

motor_4.write(0x00,size)  # size
motor_4.write(0x04,int(motor_4_duty))   # DUTY
motor_4.write(0x08,0)       # valid

motor_5.write(0x00,size)  # size
motor_5.write(0x04,int(motor_5_duty))   # DUTY
motor_5.write(0x08,0)       # valid

300300


## YOLOv3 Utility functions


In [4]:
anchor_list = [10, 14, 23, 27, 37, 58, 81, 82, 135, 169, 344, 319]
anchors = np.array(anchor_list).reshape(-1, 2)
# 클래스 정보 가져오기 함수
def get_class(classes_path):
    with open(classes_path) as f:
        class_names = f.readlines()
    class_names = [c.strip() for c in class_names]
    return class_names
    
classes_path = "/home/xilinx/jupyter_notebooks/KGIC/modelCode/configs/lane_class.txt"
class_names = get_class(classes_path)

num_classes = len(class_names)
hsv_tuples = [(1.0 * x / num_classes, 1., 1.) for x in range(num_classes)]
colors = list(map(lambda x: colorsys.hsv_to_rgb(*x), hsv_tuples))
colors = list(map(lambda x: 
                  (int(x[0] * 255), int(x[1] * 255), int(x[2] * 255)), 
                  colors))
random.seed(0)
random.shuffle(colors)
random.seed(None)

# 이미지 비율을 유지하며 padding하여 리사이즈하는 함수
def letterbox_image(image, size):
    ih, iw, _ = image.shape
    w, h = size
    scale = min(w/iw, h/ih)
    nw = int(iw*scale)
    nh = int(ih*scale)
    image = cv2.resize(image, (nw, nh), interpolation=cv2.INTER_LINEAR)
    new_image = np.ones((h, w, 3), np.uint8) * 128
    h_start = (h-nh)//2
    w_start = (w-nw)//2
    new_image[h_start:h_start+nh, w_start:w_start+nw, :] = image
    return new_image

# 이미지 전처리 함수
def pre_process(image, model_image_size):
    image = image[..., ::-1]
    image_h, image_w, _ = image.shape
    if model_image_size != (None, None):
        assert model_image_size[0] % 32 == 0, 'Multiples of 32 required'
        assert model_image_size[1] % 32 == 0, 'Multiples of 32 required'
        boxed_image = letterbox_image(image, tuple(reversed(model_image_size)))
    else:
        new_image_size = (image_w - (image_w % 32), image_h - (image_h % 32))
        boxed_image = letterbox_image(image, new_image_size)
    image_data = np.array(boxed_image, dtype='float32')
    image_data /= 255.
    image_data = np.expand_dims(image_data, 0) 	
    return image_data

def _get_feats(feats, anchors, num_classes, input_shape):
    num_anchors = len(anchors)
    anchors_tensor = np.reshape(np.array(anchors, dtype=np.float32), [1, 1, 1, num_anchors, 2])
    grid_size = np.shape(feats)[1:3]
    nu = num_classes + 5
    predictions = np.reshape(feats, [-1, grid_size[0], grid_size[1], num_anchors, nu])
    grid_y = np.tile(np.reshape(np.arange(grid_size[0]), [-1, 1, 1, 1]), [1, grid_size[1], 1, 1])
    grid_x = np.tile(np.reshape(np.arange(grid_size[1]), [1, -1, 1, 1]), [grid_size[0], 1, 1, 1])
    grid = np.concatenate([grid_x, grid_y], axis=-1)
    grid = np.array(grid, dtype=np.float32)

    # Confidence score 계산 확인
    box_xy = (1 / (1 + np.exp(-predictions[..., :2])) + grid) / np.array(grid_size[::-1], dtype=np.float32)
    box_wh = np.exp(predictions[..., 2:4]) * anchors_tensor / np.array(input_shape[::-1], dtype=np.float32)
    box_confidence = 1 / (1 + np.exp(-predictions[..., 4:5]))
    box_class_probs = 1 / (1 + np.exp(-predictions[..., 5:]))

    return box_xy, box_wh, box_confidence, box_class_probs



def correct_boxes(box_xy, box_wh, input_shape, image_shape):
    box_yx = box_xy[..., ::-1]
    box_hw = box_wh[..., ::-1]
    input_shape = np.array(input_shape, dtype = np.float32)
    image_shape = np.array(image_shape, dtype = np.float32)
    new_shape = np.around(image_shape * np.min(input_shape / image_shape))
    offset = (input_shape - new_shape) / 2. / input_shape
    scale = input_shape / new_shape
    box_yx = (box_yx - offset) * scale
    box_hw *= scale

    box_mins = box_yx - (box_hw / 2.)
    box_maxes = box_yx + (box_hw / 2.)
    boxes = np.concatenate([
        box_mins[..., 0:1],
        box_mins[..., 1:2],
        box_maxes[..., 0:1],
        box_maxes[..., 1:2]
    ], axis = -1)
    boxes *= np.concatenate([image_shape, image_shape], axis = -1)
    return boxes


def boxes_and_scores(feats, anchors, classes_num, input_shape, image_shape):
    box_xy, box_wh, box_confidence, box_class_probs = _get_feats(feats, anchors, classes_num, input_shape)
    boxes = correct_boxes(box_xy, box_wh, input_shape, image_shape)
    boxes = np.reshape(boxes, [-1, 4])
    box_scores = box_confidence * box_class_probs
    box_scores = np.reshape(box_scores, [-1, classes_num])
    return boxes, box_scores

def nms_boxes(boxes, scores):
    """Suppress non-maximal boxes.

    # Arguments
        boxes: ndarray, boxes of objects.
        scores: ndarray, scores of objects.

    # Returns
        keep: ndarray, index of effective boxes.
    """
    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    areas = (x2-x1+1)*(y2-y1+1)
    order = scores.argsort()[::-1]

    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)

        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])

        w1 = np.maximum(0.0, xx2 - xx1 + 1)
        h1 = np.maximum(0.0, yy2 - yy1 + 1)
        inter = w1 * h1

        ovr = inter / (areas[i] + areas[order[1:]] - inter)
        inds = np.where(ovr <= 0.1)[0]  # threshold
        order = order[inds + 1]

    return keep

# 모델 평가 함수: YOLOv3-Tiny의 2-출력 레이어 구조에 맞게 anchor_mask 설정
def evaluate(yolo_outputs, image_shape, class_names, anchors, max_boxes=20):
    score_thresh = 0.01
    anchor_mask = [[3, 4, 5], [0, 1, 2]]  # YOLOv3-Tiny에 맞춘 2개의 출력 레이어용 anchor mask
    boxes = []
    box_scores = []
    input_shape = np.shape(yolo_outputs[0])[1:3] * np.array([32, 32])

    for i in range(len(yolo_outputs)):
        _boxes, _box_scores = boxes_and_scores(
            yolo_outputs[i], anchors[anchor_mask[i]], len(class_names), 
            input_shape, image_shape)
        boxes.append(_boxes)
        box_scores.append(_box_scores)
    boxes = np.concatenate(boxes, axis=0)
    box_scores = np.concatenate(box_scores, axis=0)
    
    print("YOLO OUT SHAPES: ", [o.shape for o in yolo_outputs])

    mask = box_scores >= score_thresh
    boxes_ = []
    scores_ = []
    classes_ = []
    for c in range(len(class_names)):
        class_boxes_np = boxes[mask[:, c]]
        class_box_scores_np = box_scores[:, c][mask[:, c]]
        
        # Non-Max Suppression with max_boxes limit
        nms_index_np = nms_boxes(class_boxes_np, class_box_scores_np)
        nms_index_np = nms_index_np[:max_boxes]  # Limit to max_boxes
        
        class_boxes_np = class_boxes_np[nms_index_np]
        class_box_scores_np = class_box_scores_np[nms_index_np]
        classes_np = np.ones_like(class_box_scores_np, dtype=np.int32) * c
        boxes_.append(class_boxes_np)
        scores_.append(class_box_scores_np)
        classes_.append(classes_np)
        
    boxes_ = np.concatenate(boxes_, axis=0)
    scores_ = np.concatenate(scores_, axis=0)
    classes_ = np.concatenate(classes_, axis=0)

    # confidence scores 출력
    print("Confidence scores for detected boxes:", scores_)
    
    return boxes_, scores_, classes_


## DPU Setting

In [5]:
dpu = overlay.runner
inputTensors = dpu.get_input_tensors()
outputTensors = dpu.get_output_tensors()
shapeIn = tuple(inputTensors[0].dims)
shapeOut0 = (tuple(outputTensors[0].dims)) # (1, 13, 13, 75)
shapeOut1 = (tuple(outputTensors[1].dims)) # (1, 26, 26, 75)
outputSize0 = int(outputTensors[0].get_data_size() / shapeIn[0]) # 12675
outputSize1 = int(outputTensors[1].get_data_size() / shapeIn[0]) # 50700
input_data = [np.empty(shapeIn, dtype=np.float32, order="C")]
output_data = [np.empty(shapeOut0, dtype=np.float32, order="C"), 
               np.empty(shapeOut1, dtype=np.float32, order="C")]
image = input_data[0]

## Driving Funtions

In [9]:
# ================== 조향 튜닝 파라미터 (트랙에서 조정) ==================
DRIVE_SPEED = 50         # 구동 속도(0~100). 첫 테스트 안전값(사용자 지정)
REF_X = 128              # 조향 기준 x (256 리사이즈 이미지 중앙)
STEER_DIR = -1           # 차선이 오른쪽이면 우조향(+1). 트랙에서 반대로 돌면 -1 로 뒤집기
STEER_GAIN = 20.0 / 128.0  # 화면 오프셋(±128px) -> 조향 타겟(±20) 비례계수
# =====================================================================

# SPI 초기화 (조향 가변저항 피드백)
spi0 = spidev.SpiDev()
spi0.open(0, 0)
spi0.max_speed_hz = 20000000
spi0.mode = 0b00


class RobotController:
    # data_collection 검증본의 하드웨어/조향/구동 로직을 그대로 사용.
    # 차이점은 조향 타겟(steering_angle)을 키보드가 아니라 카메라(비전)가 준다는 것.
    def __init__(self):
        self.image_processor = ImageProcessor()
        self.size = 600600  # 2ms

        self.left_speed = 0
        self.right_speed = 0
        self.steering_angle = 0      # 조향 타겟(-20~20). +면 우조향 [data_collection 규약]
        self.speed = DRIVE_SPEED

        # 조향 duty 램프 (data_collection과 동일)
        self.current_duty = self.size // 2
        self.min_duty = self.size // 2
        self.max_duty = int(self.size * 0.8)
        self.duty_step = int(self.size * 0.02)
        self.last_steering_time = time.time()

        # 가변저항 실측값 (data_collection 검증본)
        self.resistance_most_left = 2338
        self.resistance_most_right = 1512

        self.stop_flag = False
        self.init_motors()

    def init_motors(self):
        for motor in [motor_0, motor_1, motor_2, motor_3, motor_4, motor_5]:
            motor.write(0x00, self.size)
            motor.write(0x04, self.min_duty)
            motor.write(0x08, 0)

    def map_value(self, x, in_min, in_max, out_min, out_max):
        if x <= in_min:
            return out_max
        elif x >= in_max:
            return out_min
        else:
            return (in_max - x) * (out_max - out_min) / (in_max - in_min) + out_min

    def read_adc(self, spi):
        r = spi.xfer2([0x00, 0x00])
        return ((r[0] & 0x0F) << 8) | r[1]

    # ---- 조향 (data_collection 그대로: motor_4=좌, motor_5=우, duty 램프) ----
    def right(self):
        now = time.time()
        if now - self.last_steering_time > 0.05:
            self.current_duty = min(self.max_duty, self.current_duty + self.duty_step)
            self.last_steering_time = now
        motor_4.write(0x08, 0)  # steering_left off
        motor_5.write(0x08, 1)  # steering_right on
        motor_5.write(0x04, self.current_duty)

    def left(self):
        now = time.time()
        if now - self.last_steering_time > 0.05:
            self.current_duty = min(self.max_duty, self.current_duty + self.duty_step)
            self.last_steering_time = now
        motor_4.write(0x08, 1)  # steering_left on
        motor_5.write(0x08, 0)  # steering_right off
        motor_4.write(0x04, self.current_duty)

    def stay(self):
        self.current_duty = self.min_duty
        motor_5.write(0x08, 0)
        motor_4.write(0x08, 0)
        motor_5.write(0x04, self.current_duty)
        motor_4.write(0x04, self.current_duty)

    # ---- 구동 (data_collection 그대로: motor_2/3=좌, motor_0/1=우) ----
    def set_left_speed(self, speed):
        duty = int(self.size * (abs(speed) / 100))
        motor_2.write(0x04, duty)
        motor_3.write(0x04, duty)
        if speed > 0:
            motor_3.write(0x08, 0)
            motor_2.write(0x08, 1)
        else:
            motor_3.write(0x08, 1)
            motor_2.write(0x08, 0)

    def set_right_speed(self, speed):
        duty = int(self.size * (abs(speed) / 100))
        motor_1.write(0x04, duty)
        motor_0.write(0x04, duty)
        if speed > 0:
            motor_1.write(0x08, 0)
            motor_0.write(0x08, 1)
        else:
            motor_1.write(0x08, 1)
            motor_0.write(0x08, 0)

    # ---- 조향 폐루프 (data_collection 그대로: ADC->±20 vs steering_angle) ----
    def control_motors(self):
        mapped_resistance = self.map_value(
            self.read_adc(spi0),
            self.resistance_most_right,
            self.resistance_most_left,
            -20, 20
        )
        if abs(mapped_resistance - self.steering_angle) < 0.5:
            self.stay()
        elif mapped_resistance > self.steering_angle:
            self.left()
        else:
            self.right()
        self.set_left_speed(self.left_speed)
        self.set_right_speed(self.right_speed)

    # ---- 비전 결과 -> 조향 타겟(±20, data_collection 규약) ----
    def vision_to_target(self, angle, center):
        if center is None:
            return 0.0   # 미검출 -> 중앙(직진)
        offset = center - REF_X                 # +면 차선이 화면 오른쪽
        target = STEER_DIR * offset * STEER_GAIN
        return float(np.clip(target, -20, 20))

    # ---- 메인 루프: 카메라 -> 비전 -> 조향타겟 -> 폐루프 ----
    def run(self, camera_index=0):
        cap = cv2.VideoCapture(camera_index)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        if not cap.isOpened():
            print("카메라를 열 수 없습니다.")
            return

        image_widget = widgets.Image(format='jpeg')
        try:
            while not self.stop_flag:
                ret, frame = cap.read()
                if not ret:
                    print("프레임을 읽을 수 없습니다.")
                    break

                angle, vis = self.image_processor.process_frame(frame)
                self.steering_angle = self.vision_to_target(angle, self.image_processor.last_lane_center)
                self.left_speed = self.speed
                self.right_speed = self.speed
                self.control_motors()

                ok, jpeg = cv2.imencode('.jpeg', vis)
                if ok:
                    image_widget.value = jpeg.tobytes()
                    display(image_widget)
                    clear_output(wait=True)
        except KeyboardInterrupt:
            print("사용자 중지 (Ctrl+C)")
        finally:
            self.stop_flag = True
            cap.release()
            self.cleanup()

    def cleanup(self):
        for motor in [motor_0, motor_1, motor_2, motor_3, motor_4, motor_5]:
            motor.write(0x00, 0)
            motor.write(0x04, 0)
            motor.write(0x08, 0)
        clear_output(wait=True)
        print("모터 정지 및 종료.")


print("RobotController(data_collection based, autonomous) ready")


RobotController(data_collection based, autonomous) ready


## ImageProcessor

In [7]:
class ImageProcessor:
    # 비전 파이프라인: BEV(학습 전처리와 동일) -> ROI -> 256 리사이즈 -> DPU -> 차선검출 -> 각도
    def __init__(self):
        self.point_detection_height = 20   # 검출점 y (256 리사이즈 기준)
        self.reference_point_x = 200       # 각도 기준점 x (canonical image_processor.py 값)
        self.reference_point_y = 240
        self.last_lane_center = None       # 최근 검출된 차선 중심 x (조향 타겟 계산용)

    def roi_rectangle_below(self, img, cutting_idx):
        return img[cutting_idx:, :]

    def warpping(self, image, srcmat, dstmat):
        h, w = image.shape[0], image.shape[1]
        M = cv2.getPerspectiveTransform(srcmat, dstmat)
        minv = cv2.getPerspectiveTransform(dstmat, srcmat)
        warped = cv2.warpPerspective(image, M, (w, h))
        return warped, minv

    def bird_convert(self, img, srcmat, dstmat):
        srcmat = np.float32(srcmat)
        dstmat = np.float32(dstmat)
        warped, _ = self.warpping(img, srcmat, dstmat)
        return warped

    def calculate_angle(self, x1, y1, x2, y2):
        if x1 == x2:
            return 90.0
        slope = (y2 - y1) / (x2 - x1)
        return -math.degrees(math.atan(slope))

    def detect_lane_center_x(self, xyxy_results):
        # 가장 오른쪽 레이블(차선)의 중심 X (canonical과 동일 로직)
        rx_min = None; rx_max = None; rightmost = -float('inf')
        for box in xyxy_results:
            y1, x1, y2, x2 = box
            if x1 > rightmost:
                rightmost = x1
                rx_min = int(x1); rx_max = int(x2)
        if rx_min is not None:
            return (rx_min + rx_max) // 2
        return None

    def process_frame(self, img):
        h, w = img.shape[0], img.shape[1]
        # (1) BEV 변환 - 학습 전처리와 동일 (driving/image_processor.py 실측 좌표)
        src_mat = [[238, 316], [402, 313], [501, 476], [155, 476]]
        dst_mat = [[round(w * 0.3), 0], [round(w * 0.7), 0],
                   [round(w * 0.7), h], [round(w * 0.3), h]]
        bird_img = self.bird_convert(img, src_mat, dst_mat)

        # (2) ROI 하단 추출(cutting_idx=300, 캐노니컬과 동일) + 256 리사이즈
        roi_image = self.roi_rectangle_below(bird_img, cutting_idx=300)
        resized_img = cv2.resize(roi_image, (256, 256))

        # (3) 전처리 + DPU 추론
        image_size = resized_img.shape[:2]
        image_data = np.array(pre_process(resized_img, (256, 256)), dtype=np.float32)
        start_time = time.time()
        image[0, ...] = image_data.reshape(shapeIn[1:])
        job_id = dpu.execute_async(input_data, output_data)
        dpu.wait(job_id)
        end_time = time.time()

        conv_out0 = np.reshape(output_data[0], shapeOut0)
        conv_out1 = np.reshape(output_data[1], shapeOut1)
        yolo_outputs = [conv_out0, conv_out1]
        boxes, scores, classes = evaluate(yolo_outputs, image_size, class_names, anchors)
        
        print(float(conv_out0.min()), float(conv_out0.max()))

        # 시각화(박스)
        for box in boxes:
            cv2.rectangle(resized_img, (int(box[1]), int(box[0])),
                          (int(box[3]), int(box[2])), (0, 255, 0), 2)

        # (4) 차선 중심 + 각도
        center = self.detect_lane_center_x(boxes)
        self.last_lane_center = center
        if center is None:
            print("차선 미검출 -> 조향 중립(직진) 유지")
            return 90.0, resized_img

        angle = self.calculate_angle(self.reference_point_x, self.reference_point_y,
                                     center, self.point_detection_height)
        # 시각화(라인/중심점)
        cv2.line(resized_img, (self.reference_point_x, self.reference_point_y),
                 (center, self.point_detection_height), (0, 0, 255), 3)
        cv2.circle(resized_img, (center, self.point_detection_height), 6, (0, 255, 255), -1)
        exec_ms = (end_time - start_time) * 1000.0
        fps = 1000.0 / exec_ms if exec_ms > 0 else 0.0
        print(f"angle={angle:.1f} center={center} FPS={fps:.1f}")
        return angle, resized_img


print("ImageProcessor(BEV) ready")


ImageProcessor(BEV) ready


## Driving!

In [8]:
# ===== 실행 =====
# 주의: 첫 테스트는 반드시 바퀴를 지면에서 띄운 상태로!
#  - 좌/우 바퀴가 같은 방향(전진)으로 도는지
#  - 차선이 화면 오른쪽이면 우조향인지 (반대면 위 셀의 STEER_DIR 을 -1 로)
controller = RobotController()
controller.run(camera_index=0)


모터 정지 및 종료.
